# Telepati 8.0: AI AgriData Intelligence Race
## Deteksi Penyakit Daun Padi Multi-Class (11 Kelas Kanonikal)
---
### Bagian A — Setup & Preprocessing
Instalasi dependensi, unduh dataset, audit anotasi COCO, konversi ke format YOLO, dan verifikasi distribusi kelas.


In [1]:
%pip install gdown matplotlib ultralytics pandas numpy pillow

In [2]:
!gdown "1CAPmwVmYW6499Ic1wx8TFptLV8beDZcX" -O dataset.zip

Downloading...
From (original): https://drive.google.com/uc?id=1CAPmwVmYW6499Ic1wx8TFptLV8beDZcX
From (redirected): https://drive.google.com/uc?id=1CAPmwVmYW6499Ic1wx8TFptLV8beDZcX&confirm=t&uuid=44411f4b-979b-40cf-b435-bca14ecbd20b
To: /kaggle/working/dataset.zip
100%|█████████████████████████████████████████| 620M/620M [00:05<00:00, 113MB/s]


In [3]:
!unzip -q -o dataset.zip -d dataset/


## A.1 Import Library & Kunci Determinisme
Mengimpor modul penting serta mengunci determinisme 4-lapis (Python, NumPy, PyTorch CPU, CUDA, dan CuDNN) untuk kepatuhan Audit Run juri.


In [4]:
import json
import os
import random
import shutil
import zipfile
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

set_seed(42)


## A.2 Konfigurasi Global (`Config`)
Konfigurasi jalur path hibrida (Kaggle vs Lokal), pemetaan 11 kelas kanonikal, serta hyperparameter pelatihan model from scratch.


In [5]:
@dataclass
class Config:
    # --- Deteksi Lingkungan Kerja (Kaggle vs Lokal) ---
    is_kaggle: bool = field(default_factory=lambda: Path("/kaggle/working").exists())

    # --- Path dataset & output ---
    dataset_root: Path = field(
        default_factory=lambda: (
            Path(r"/kaggle/working/dataset/Telepati 8.0 Datasets")
            if Path("/kaggle/working").exists()
            else Path.cwd() / "dataset" / "Telepati 8.0 Datasets"
        )
    )
    yolo_out: Path = field(
        default_factory=lambda: (
            Path(r"/kaggle/working/train")
            if Path("/kaggle/working").exists()
            else Path.cwd() / "train"
        )
    )
    coco_annotation_file: str = "_annotations.coco.json"

    # --- Split dataset ---
    coco_splits: List[str] = field(default_factory=lambda: ["train", "valid", "test"])
    split_mapping: Dict[str, str] = field(
        default_factory=lambda: {"train": "train", "valid": "val", "test": "test"}
    )

    # --- Reprodusibilitas ---
    seed: int = 42

    # --- Mapping & daftar kelas (11 Kelas Kanonikal Resmi) ---
    class_mapping: Dict[str, str] = field(
        default_factory=lambda: {
            "Bacterial leaf blight": "bacterial_leaf_blight",
            "Bacterial panicle Blight": "bacterial_panicle_blight",
            "Blast": "blast",
            "Leaf blast": "blast",
            "Infected Blast": "blast",
            "Brown spot": "brown_spot",
            "BrownSpot": "brown_spot",
            "False-Smut": "false_smut",
            "Healthy Rice Leaf": "healthy",
            "Healthy Rice beads": "healthy",
            "Healthy": "healthy",
            "healthy": "healthy",
            "Leaf-roller": "leaf_roller",
            "Leaf Scald": "leaf_scald",
            "Leaf scald": "leaf_scald",
            "Narrow brown": "narrow_brown",
            "Sheath Blight": "sheath_blight",
            "Rice-Tungro": "tungro",
        }
    )
    class_names: List[str] = field(
        default_factory=lambda: [
            "bacterial_leaf_blight",
            "bacterial_panicle_blight",
            "blast",
            "brown_spot",
            "false_smut",
            "healthy",
            "leaf_roller",
            "leaf_scald",
            "narrow_brown",
            "sheath_blight",
            "tungro",
        ]
    )

    # --- Hyperparameter training YOLO26s (From Scratch) ---
    model_yaml: str = "yolo26s-p2.yaml"
    train_epochs: int = 200
    train_batch: int = 32
    train_imgsz: int = 640
    train_save_period: int = 50
    train_pretrained: bool = False
    train_hsv_h: float = 0.01
    train_hsv_s: float = 0.3
    train_hsv_v: float = 0.3
    train_translate: float = 0.2
    train_shear: float = 0.0
    train_perspective: float = 0.0
    train_scale: float = 0.4
    train_degrees: float = 0.0
    train_fliplr: float = 0.3
    train_flipud: float = 0.3
    train_mosaic: float = 0.4
    train_close_mosaic: int = 10
    train_erasing: float = 0.1
    train_auto_augment: Optional[str] = None
    train_cos_lr: bool = True
    train_patience: int = 0
    train_device: any = field(
        default_factory=lambda: [0, 1] if Path("/kaggle/working").exists() else ""
    )
    train_project: Path = field(
        default_factory=lambda: (
            Path(r"/kaggle/working/train")
            if Path("/kaggle/working").exists()
            else Path.cwd() / "train"
        )
    )
    train_name: str = "train_yolo26s_p2_bifpn"

    @property
    def class_to_id(self) -> Dict[str, int]:
        return {name: i for i, name in enumerate(self.class_names)}

    @property
    def best_weights_path(self) -> str:
        candidates = [
            Path(self.train_project) / self.train_name / "weights" / "best.pt",
            Path("/kaggle/working/runs/detect/kaggle/working/train") / self.train_name / "weights" / "best.pt",
            Path.cwd() / "runs" / "detect" / self.train_name / "weights" / "best.pt",
        ]
        for c in candidates:
            if c.exists():
                return str(c)
        return str(candidates[0])

CFG = Config()
set_seed(CFG.seed)


## A.3 Cek Ketersediaan Folder & File Anotasi COCO
Memverifikasi keberadaan split dataset (train, valid, test) dan file anotasi `_annotations.coco.json`.


In [6]:
def check_dataset_structure(cfg: Config) -> None:
    for split in cfg.coco_splits:
        split_dir = cfg.dataset_root / split
        json_path = split_dir / cfg.coco_annotation_file

        print(f"\n[{split}]")
        print("Folder :", split_dir)
        print("JSON   :", json_path)
        print("JSON exists :", json_path.exists())


check_dataset_structure(CFG)


[train]
Folder : /kaggle/working/dataset/Telepati 8.0 Datasets/train
JSON   : /kaggle/working/dataset/Telepati 8.0 Datasets/train/_annotations.coco.json
JSON exists : True

[valid]
Folder : /kaggle/working/dataset/Telepati 8.0 Datasets/valid
JSON   : /kaggle/working/dataset/Telepati 8.0 Datasets/valid/_annotations.coco.json
JSON exists : True

[test]
Folder : /kaggle/working/dataset/Telepati 8.0 Datasets/test
JSON   : /kaggle/working/dataset/Telepati 8.0 Datasets/test/_annotations.coco.json
JSON exists : True


## A.4 Audit Anotasi COCO Dataset
Menginspeksi validitas bounding box, mendeteksi bbox di luar resolusi citra, serta menghitung frekuensi kategori mentah.


In [7]:
def audit_coco_json(json_path: Path) -> dict:
    data = json.loads(json_path.read_text(encoding="utf-8"))

    images = {img["id"]: img for img in data["images"]}
    categories = {cat["id"]: cat["name"] for cat in data["categories"]}

    anotasi_per_image = {}
    for anotasi in data["annotations"]:
        anotasi_per_image.setdefault(anotasi["image_id"], []).append(anotasi)

    gambar_tanpa_anotasi = [
        gambar
        for id_gambar, gambar in images.items()
        if id_gambar not in anotasi_per_image
    ]

    invalid_bbox = []
    for anotasi in data["annotations"]:
        if anotasi["image_id"] not in images:
            continue

        img = images[anotasi["image_id"]]
        W = img["width"]
        H = img["height"]
        bbox = anotasi.get("bbox")

        if bbox is None or len(bbox) != 4 or bbox[2] <= 0 or bbox[3] <= 0:
            invalid_bbox.append(anotasi)
            continue

        x, y, w, h = bbox
        if x < 0 or y < 0 or x + w > W or y + h > H:
            invalid_bbox.append(anotasi)

    class_counts = Counter()
    for anotasi in data["annotations"]:
        nama = categories.get(anotasi["category_id"], "UNKNOWN")
        class_counts[nama] += 1

    print(f"Gambar              : {len(images)}")
    print(f"Anotasi            : {len(data['annotations'])}")
    print(f"Gambar tanpa anotasi : {len(gambar_tanpa_anotasi)}")
    print(f"Bbox invalid/outside    : {len(invalid_bbox)}")
    print("Distribusi kategori:")
    for name, count in class_counts.most_common():
        print(f"{name:<35}: {count}")

    return data


def run_coco_audit(cfg: Config) -> Dict[str, dict]:
    data_coco = {}
    for split in cfg.coco_splits:
        json_path = cfg.dataset_root / split / cfg.coco_annotation_file
        print(f"AUDIT {split.upper()}")
        data_coco[split] = audit_coco_json(json_path)
    return data_coco


data_coco = run_coco_audit(CFG)

AUDIT TRAIN
Gambar              : 10133
Anotasi            : 20163
Gambar tanpa anotasi : 59
Bbox invalid/outside    : 427
Distribusi kategori:
BrownSpot                          : 3388
Blast                              : 3033
Rice-Tungro                        : 2540
Sheath Blight                      : 1762
Brown spot                         : 1622
Leaf Scald                         : 1224
False-Smut                         : 852
Infected Blast                     : 832
Leaf-roller                        : 812
Healthy Rice Leaf                  : 717
Healthy Rice beads                 : 699
Healthy                            : 686
Bacterial panicle Blight           : 528
Bacterial leaf blight              : 476
Leaf blast                         : 284
healthy                            : 272
Narrow brown                       : 222
Leaf scald                         : 214
AUDIT VALID
Gambar              : 2106
Anotasi            : 4888
Gambar tanpa anotasi : 13
Bbox invalid/outside 

## A.5 Pembuatan Struktur Direktori YOLO
Membuat hierarki folder `images/{train,val,test}` dan `labels/{train,val,test}`.


In [8]:
def create_yolo_dirs(cfg: Config) -> None:
    for split in ["train", "val", "test"]:
        (cfg.yolo_out / "images" / split).mkdir(parents=True, exist_ok=True)
        (cfg.yolo_out / "labels" / split).mkdir(parents=True, exist_ok=True)

    print("Folder YOLO berhasil dibuat.")


create_yolo_dirs(CFG)

Folder YOLO berhasil dibuat.


## A.6 Konversi COCO ke Format YOLO & Pemetaan 11 Kelas Kanonikal
Normalisasi koordinat bbox `(xc, yc, w, h)`, pembersihan bbox invalid, serta standarisasi 18 label mentah ke 11 kelas resmi.


In [9]:
def convert_coco_to_yolo(cfg: Config) -> None:
    class_to_id = cfg.class_to_id

    for split in cfg.coco_splits:
        input_dir = cfg.dataset_root / split
        json_path = input_dir / cfg.coco_annotation_file

        output_split = "val" if split == "valid" else split
        image_out = cfg.yolo_out / "images" / output_split
        label_out = cfg.yolo_out / "labels" / output_split

        with open(json_path, "r", encoding="utf-8") as f:
            coco = json.load(f)

        images = {image["id"]: image for image in coco["images"]}
        categories = {
            category["id"]: category["name"] for category in coco["categories"]
        }

        annotations_per_image = {image_id: [] for image_id in images}

        # dipakai untuk melacak kelas apa saja yang ada di gambar yang akhirnya di-skip
        raw_categories_per_image = {image_id: [] for image_id in images}

        invalid_count = 0
        clipped_count = 0
        unmapped_count = 0

        invalid_class_counts = Counter()
        clipped_class_counts = Counter()
        unmapped_class_counts = Counter()

        for ann in coco["annotations"]:
            image_id = ann["image_id"]
            category_name = categories[ann["category_id"]]
            raw_categories_per_image[image_id].append(category_name)

            mapped_class = cfg.class_mapping.get(category_name)

            if mapped_class is None:
                unmapped_count += 1
                unmapped_class_counts[category_name] += 1
                continue

            class_id = class_to_id[mapped_class]

            x, y, w, h = ann["bbox"]
            img_w = images[image_id]["width"]
            img_h = images[image_id]["height"]

            x1, y1, x2, y2 = x, y, x + w, y + h

            new_x1 = max(0, x1)
            new_y1 = max(0, y1)
            new_x2 = min(img_w, x2)
            new_y2 = min(img_h, y2)

            new_w = new_x2 - new_x1
            new_h = new_y2 - new_y1

            if new_w <= 0 or new_h <= 0:
                invalid_count += 1
                invalid_class_counts[category_name] += 1
                continue

            if new_x1 != x1 or new_y1 != y1 or new_x2 != x2 or new_y2 != y2:
                clipped_count += 1
                clipped_class_counts[category_name] += 1

            x_center = (new_x1 + new_x2) / 2 / img_w
            y_center = (new_y1 + new_y2) / 2 / img_h
            width = new_w / img_w
            height = new_h / img_h

            annotations_per_image[image_id].append(
                f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
            )

        # counter untuk gambar yang dimasukkan vs di-skip
        included_images = 0
        skipped_images = 0
        skipped_no_annotation = 0
        skipped_all_filtered = 0
        skipped_class_counts = Counter()

        for image_id, image_info in images.items():
            if not annotations_per_image[image_id]:
                skipped_images += 1

                raw_classes = raw_categories_per_image[image_id]
                if not raw_classes:
                    skipped_no_annotation += 1
                else:
                    skipped_all_filtered += 1
                    for c in raw_classes:
                        skipped_class_counts[c] += 1
                continue

            file_name = image_info["file_name"]
            source_image = input_dir / file_name

            if not source_image.exists():
                continue

            shutil.copy2(source_image, image_out / Path(file_name).name)

            label_name = Path(file_name).stem + ".txt"
            label_path = label_out / label_name

            with open(label_path, "w") as f:
                for line in annotations_per_image[image_id]:
                    f.write(line + "\n")

            included_images += 1

        # ================= RINGKASAN =================
        print(f"\n{'='*50}")
        print(f"  SPLIT: {split.upper()}")
        print(f"{'='*50}")

        print(f"\nTotal gambar                : {len(images)}")
        print(f"Gambar dimasukkan ke YOLO   : {included_images}")
        print(f"Gambar di-skip (total)      : {skipped_images}")
        print(f"  - tanpa annotation sama sekali : {skipped_no_annotation}")
        print(f"  - semua annotation dibuang     : {skipped_all_filtered}")

        print(f"\nBbox clipped   : {clipped_count}")
        print(f"Bbox invalid   : {invalid_count}")
        print(f"Bbox unmapped  : {unmapped_count}")

        print("\n-- Kelas pada gambar yang di-skip --")
        if skipped_class_counts:
            for name, count in skipped_class_counts.most_common():
                print(f"  {name:<35}: {count}")
        else:
            print("  (tidak ada / semua gambar skip memang tanpa annotation)")

        print("\n-- Kelas & jumlah bbox invalid --")
        if invalid_class_counts:
            for name, count in invalid_class_counts.most_common():
                print(f"  {name:<35}: {count}")
        else:
            print("  (tidak ada)")

        print("\n-- Kelas & jumlah bbox clipped --")
        if clipped_class_counts:
            for name, count in clipped_class_counts.most_common():
                print(f"  {name:<35}: {count}")
        else:
            print("  (tidak ada)")

        print("\n-- Kelas & jumlah bbox unmapped (tidak ada di CLASS_MAPPING) --")
        if unmapped_class_counts:
            for name, count in unmapped_class_counts.most_common():
                print(f"  {name:<35}: {count}")
        else:
            print("  (tidak ada)")


convert_coco_to_yolo(CFG)


  SPLIT: TRAIN

Total gambar                : 10133
Gambar dimasukkan ke YOLO   : 10074
Gambar di-skip (total)      : 59
  - tanpa annotation sama sekali : 59
  - semua annotation dibuang     : 0

Bbox clipped   : 427
Bbox invalid   : 0
Bbox unmapped  : 0

-- Kelas pada gambar yang di-skip --
  (tidak ada / semua gambar skip memang tanpa annotation)

-- Kelas & jumlah bbox invalid --
  (tidak ada)

-- Kelas & jumlah bbox clipped --
  False-Smut                         : 96
  Leaf-roller                        : 88
  Healthy Rice Leaf                  : 65
  Bacterial panicle Blight           : 63
  Infected Blast                     : 50
  Healthy Rice beads                 : 46
  Healthy                            : 10
  Leaf Scald                         : 4
  Bacterial leaf blight              : 3
  BrownSpot                          : 1
  Brown spot                         : 1

-- Kelas & jumlah bbox unmapped (tidak ada di CLASS_MAPPING) --
  (tidak ada)

  SPLIT: VALID

Total gam

## A.7 Pembuatan File Konfigurasi `data.yaml`
Menulis konfigurasi `data.yaml` yang menghubungkan Ultralytics ke direktori dataset hasil konversi.


In [10]:
def write_data_yaml(cfg: Config) -> str:
    yaml_text = f"""
path: {cfg.yolo_out}

train: images/train
val: images/val
test: images/test

nc: {len(cfg.class_names)}

names:
"""
    for i, name in enumerate(cfg.class_names):
        yaml_text += f"  {i}: {name}\n"

    with open(cfg.yolo_out / "data.yaml", "w") as f:
        f.write(yaml_text)

    print(yaml_text)
    return yaml_text


write_data_yaml(CFG)


path: /kaggle/working/train

train: images/train
val: images/val
test: images/test

nc: 11

names:
  0: bacterial_leaf_blight
  1: bacterial_panicle_blight
  2: blast
  3: brown_spot
  4: false_smut
  5: healthy
  6: leaf_roller
  7: leaf_scald
  8: narrow_brown
  9: sheath_blight
  10: tungro



'\npath: /kaggle/working/train\n\ntrain: images/train\nval: images/val\ntest: images/test\n\nnc: 11\n\nnames:\n  0: bacterial_leaf_blight\n  1: bacterial_panicle_blight\n  2: blast\n  3: brown_spot\n  4: false_smut\n  5: healthy\n  6: leaf_roller\n  7: leaf_scald\n  8: narrow_brown\n  9: sheath_blight\n  10: tungro\n'

## A.8 Verifikasi Sinkronisasi Citra & Label
Memvalidasi keselarasan jumlah file citra dan file label pada seluruh split dataset.


In [11]:
def verify_yolo_dataset(cfg: Config) -> None:
    for split in ["train", "val", "test"]:
        images = list((cfg.yolo_out / "images" / split).glob("*"))
        labels = list((cfg.yolo_out / "labels" / split).glob("*.txt"))
        print(split, "images:", len(images), "labels:", len(labels))


def show_sample_label(cfg: Config, split: str = "train") -> None:
    label_files = list((cfg.yolo_out / "labels" / split).glob("*.txt"))
    print(label_files[0])
    print()
    print(label_files[0].read_text())


verify_yolo_dataset(CFG)
show_sample_label(CFG, split="train")

train images: 10074 labels: 10074
val images: 2093 labels: 2093
test images: 1054 labels: 1054
/kaggle/working/train/labels/train/IMG-20241021-WA0121_jpg.rf.47b33a0d9ca4f0f53bf6438cb707598a.txt

2 0.444896 0.549343 0.224167 0.898686



## A.9 Visualisasi Sampel Ground Truth
Menampilkan sampel citra dan bounding box untuk konfirmasi visual hasil pra-pemrosesan.


In [12]:
def visualisasi_sampel(
    output_dir: Path, split: str = "train", sample_size: int = 20, cfg: Config = CFG
) -> None:
    images_dir = output_dir / "images" / split
    labels_dir = output_dir / "labels" / split

    image_files = [
        p
        for p in images_dir.rglob("*")
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]

    random.seed(cfg.seed)

    samples = random.sample(image_files, min(sample_size, len(image_files)))

    cols = 3
    rows = int(np.ceil(len(samples) / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))

    axes = np.array(axes).reshape(-1)

    for ax, image_path in zip(axes, samples):
        image = Image.open(image_path)

        W, H = image.size

        ax.imshow(image)

        label_path = (labels_dir / image_path.relative_to(images_dir)).with_suffix(
            ".txt"
        )

        # Kalau label tidak ada
        if not label_path.exists():
            ax.set_title(image_path.name)
            ax.axis("off")
            continue

        lines = label_path.read_text(encoding="utf-8").splitlines()

        for line in lines:
            if not line.strip():
                continue

            class_id, xc, yc, w, h = map(float, line.split())

            class_id = int(class_id)

            # YOLO normalized -> pixel
            xc *= W
            yc *= H
            w *= W
            h *= H

            x1 = xc - w / 2
            y1 = yc - h / 2

            rect = plt.Rectangle((x1, y1), w, h, fill=False, linewidth=2)

            ax.add_patch(rect)

            ax.text(
                x1, y1, cfg.class_names[class_id], fontsize=8, backgroundcolor="white"
            )

        ax.set_title(image_path.name)

        ax.axis("off")

    for ax in axes[len(samples) :]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


visualisasi_sampel(CFG.yolo_out, split="train", sample_size=20)

## A.10 Analisis Distribusi Kelas
Menghitung dan memetakan jumlah instans per kelas pada train, val, dan test untuk mendeteksi *class imbalance*.


In [13]:
def compute_class_distribution(cfg: Config):
    import pandas as pd

    distribution = {}

    for split in ["train", "val", "test"]:
        label_dir = cfg.yolo_out / "labels" / split

        counter = Counter()

        for label_path in label_dir.glob("*.txt"):
            lines = label_path.read_text(encoding="utf-8").splitlines()

            for line in lines:
                if not line.strip():
                    continue

                class_id = int(line.split()[0])

                counter[class_id] += 1

        distribution[split] = counter

    df = pd.DataFrame(
        {
            "class": cfg.class_names,
            "train": [distribution["train"][i] for i in range(len(cfg.class_names))],
            "val": [distribution["val"][i] for i in range(len(cfg.class_names))],
            "test": [distribution["test"][i] for i in range(len(cfg.class_names))],
        }
    )

    df["total"] = df["train"] + df["val"] + df["test"]

    df = df.sort_values("total", ascending=False).reset_index(drop=True)

    print(df)
    return df


class_distribution_df = compute_class_distribution(CFG)

                       class  train   val  test  total
0                 brown_spot   5010  1461   840   7311
1                      blast   4149   947   535   5631
2                     tungro   2540   760   411   3711
3                    healthy   2374   424   184   2982
4              sheath_blight   1762   488   269   2519
5                 leaf_scald   1438   390   222   2050
6                 false_smut    852    86    43    981
7                leaf_roller    812    87    44    943
8      bacterial_leaf_blight    476   125    59    660
9   bacterial_panicle_blight    528    45    27    600
10              narrow_brown    222    75    36    333


---
### Bagian B — Pelatihan Model (From Scratch)
Inisialisasi model YOLO dari konfigurasi YAML murni tanpa bobot pra-latih (*transfer learning dilarang*).


## B.1 Rekayasa Arsitektur Kustom (P2 High-Resolution Head & BiFPN Fusion)
Untuk mengatasi kendala kegagalan deteksi objek mikro (`brown_spot` Recall 27.1%) dan lesi memanjang (`bacterial_leaf_blight`), kami mengintegrasikan:
1. **High-Resolution Detection Head P2 (160×160)** via `yolo26s-p2.yaml` yang menyadap fitur resolusi tinggi ber-stride 4 dari backbone.
2. **BiFPN Fast Normalized Feature Fusion (`BiFPN_Concat`)** dengan parameter bobot terpelajari $w_i \ge 0$:
   $$O = \text{Concat}\left(\frac{w_1}{\epsilon + w_1 + w_2} \cdot x_1, \frac{w_2}{\epsilon + w_1 + w_2} \cdot x_2\right)$$


In [ ]:
import torch
import torch.nn as nn
import ultralytics.nn.modules as modules
import ultralytics.nn.tasks as tasks

class BiFPN_Concat(modules.Concat):
    """BiFPN Fast Normalized Feature Fusion Module."""
    def __init__(self, dimension=1):
        super().__init__(dimension=dimension)
        self.w = nn.Parameter(torch.ones(2, dtype=torch.float32), requires_grad=True)
        self.epsilon = 1e-4

    def forward(self, x):
        if len(x) == 2:
            w = torch.relu(self.w)
            norm_w = w / (torch.sum(w) + self.epsilon)
            return torch.cat([norm_w[0] * x[0], norm_w[1] * x[1]], self.d)
        return super().forward(x)

# Registrasi ke engine Ultralytics secara in-memory (mematuhi aturan single notebook)
modules.Concat = BiFPN_Concat
tasks.Concat = BiFPN_Concat
print("BiFPN_Concat successfully injected into Ultralytics engine!")


## B.2 Injeksi Fungsi Kerugian EIoU (Efficient IoU Loss)
Menggantikan penalti rasio aspek CIoU dengan **EIoU Loss** yang mendekopel penalti lebar dan tinggi secara independen:
$$\mathcal{L}_{EIoU} = 1 - IoU + \frac{\rho^2(b, b^{gt})}{c^2} + \frac{\rho^2(w, w^{gt})}{C_w^2} + \frac{\rho^2(h, h^{gt})}{C_h^2}$$
Formulasi ini mencegah distorsi gradien pada lesi daun asimetris tipis memanjang.


In [ ]:
import ultralytics.utils.loss as loss_module

def bbox_iou_eiou(box1, box2, xywh=True, GIoU=False, DIoU=False, CIoU=False, eps=1e-7):
    """Calculates EIoU Loss with decoupled width/height penalties."""
    if xywh:
        b1_x1, b1_x2 = box1[..., 0] - box1[..., 2] / 2, box1[..., 0] + box1[..., 2] / 2
        b1_y1, b1_y2 = box1[..., 1] - box1[..., 3] / 2, box1[..., 1] + box1[..., 3] / 2
        b2_x1, b2_x2 = box2[..., 0] - box2[..., 2] / 2, box2[..., 0] + box2[..., 2] / 2
        b2_y1, b2_y2 = box2[..., 1] - box2[..., 3] / 2, box2[..., 1] + box2[..., 3] / 2
        w1, h1 = box1[..., 2], box1[..., 3]
        w2, h2 = box2[..., 2], box2[..., 3]
    else:
        b1_x1, b1_y1, b1_x2, b1_y2 = box1.chunk(4, -1)
        b2_x1, b2_y1, b2_x2, b2_y2 = box2.chunk(4, -1)
        w1, h1 = b1_x2 - b1_x1, b1_y2 - b1_y1
        w2, h2 = b2_x2 - b2_x1, b2_y2 - b2_y1

    inter = (b1_x2.minimum(b2_x2) - b1_x1.maximum(b2_x1)).clamp_(0) * (
        b1_y2.minimum(b2_y2) - b1_y1.maximum(b2_y1)
    ).clamp_(0)
    union = w1 * h1 + w2 * h2 - inter + eps
    iou = inter / union

    cw = b1_x2.maximum(b2_x2) - b1_x1.minimum(b2_x1)
    ch = b1_y2.maximum(b2_y2) - b1_y1.minimum(b2_y1)
    c2 = cw.pow(2) + ch.pow(2) + eps
    rho2 = ((b2_x1 + b2_x2 - b1_x1 - b1_x2).pow(2) + (b2_y1 + b2_y2 - b1_y1 - b1_y2).pow(2)) / 4

    rho_w2 = (w1 - w2).pow(2)
    rho_h2 = (h1 - h2).pow(2)
    cw2 = cw.pow(2) + eps
    ch2 = ch.pow(2) + eps

    eiou = iou - (rho2 / c2 + rho_w2 / cw2 + rho_h2 / ch2)
    return eiou

loss_module.bbox_iou = bbox_iou_eiou
print("EIoU Bounding Box Loss successfully injected into BboxLoss!")


## B.3 Inisialisasi Bobot Kaiming Normal & Pelatihan Model (From Scratch)
Dalam rezim pelatihan dari nol (*train from scratch*), inisialisasi bobot standar acak sering memicu *vanishing gradients*. Kami menerapkan inisialisasi Kaiming Normal (He et al.) untuk stabilitas konvergensi awal.


In [ ]:
def init_kaiming_weights(model):
    """Inisialisasi Kaiming Normal pada seluruh konvolusi dan linear layers."""
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.constant_(m.weight, 1)
            nn.init.constant_(m.bias, 0)
    print("Kaiming Normal initialization applied to model weights!")


In [ ]:
from ultralytics import YOLO

def train_model(cfg: Config):
    # 1. Inisialisasi model kosong via arsitektur YAML (No Pretrained Weights)
    model = YOLO(cfg.model_yaml)
    init_kaiming_weights(model.model)

    # 2. Eksekusi Pelatihan dengan AdamW + Cosine Annealing
    results = model.train(
        data=str(cfg.yolo_out / "data.yaml"),
        epochs=cfg.train_epochs,
        batch=cfg.train_batch,
        imgsz=cfg.train_imgsz,
        save_period=cfg.train_save_period,
        pretrained=cfg.train_pretrained,
        seed=cfg.seed,
        deterministic=True,
        optimizer="AdamW",
        cos_lr=cfg.train_cos_lr,
        patience=cfg.train_patience,
        # HSV Augmentation
        hsv_h=cfg.train_hsv_h,
        hsv_s=cfg.train_hsv_s,
        hsv_v=cfg.train_hsv_v,
        # Geometric Augmentation
        translate=cfg.train_translate,
        shear=cfg.train_shear,
        perspective=cfg.train_perspective,
        scale=cfg.train_scale,
        degrees=cfg.train_degrees,
        # Flip & Mosaic
        fliplr=cfg.train_fliplr,
        flipud=cfg.train_flipud,
        mosaic=cfg.train_mosaic,
        close_mosaic=cfg.train_close_mosaic,
        erasing=cfg.train_erasing,
        auto_augment=cfg.train_auto_augment,
        project=cfg.train_project,
        name=cfg.train_name,
        device=cfg.train_device,
        exist_ok=True,
    )
    return model, results

# Jalankan training (eksekusi di Kaggle GPU)
# model_trained, results_trained = train_model(CFG)


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.163 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=None, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/train/data.yaml, degrees=0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasin

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      2/200      5.79G       3.16      6.106    0.06364         32        640: 100% ━━━━━━━━━━━━ 315/315 3.0it/s 1:45
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 33/33 3.1it/s 10.5s
                   all       2093       4887       0.23       0.07     0.0219    0.00607

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      3/200      5.81G      2.586      3.975    0.04735         46        640: 100% ━━━━━━━━━━━━ 315/315 3.2it/s 1:39
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 33/33 3.1it/s 10.6s
                   all       2093       4887      0.344      0.147     0.0606     0.0206

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      4/200      5.81G      2.361      3.582    0.03987         36        640: 100% ━━━━━━━━━

---
### Bagian C — Evaluasi Model & Verifikasi Metrik
Evaluasi komprehensif pada set validasi dan test set menggunakan metrik mAP@50 dan F1-Score.


## C.1 Memuat Bobot Model Terbaik (`best.pt`)


In [15]:
def load_best_model(weights_path: str) -> YOLO:
    return YOLO(weights_path)


best_model = load_best_model(CFG.best_weights_path)

## C.2 Evaluasi pada Data Validasi (Validation Set)


In [16]:
def evaluate_model(model: YOLO, cfg: Config, split: str):
    metrics = model.val(
        data=str(cfg.yolo_out / "data.yaml"), split=split, imgsz=cfg.train_imgsz
    )
    return metrics


metrics = evaluate_model(best_model, CFG, split="val")

print("mAP50 :", metrics.box.map50)
print("mAP50-95 :", metrics.box.map)

Ultralytics 8.4.163 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26s summary (fused): 120 layers, 9,469,437 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1373.3±557.4 MB/s, size: 48.6 KB)
val: Scanning /kaggle/working/train/labels/val.cache... 2093 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2093/2093 93.4Mit/s 0.0s
val: /kaggle/working/train/images/val/rtf_38_jpg.rf.2131d44635500a76448ff693d761baa1.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 131/131 3.9it/s 33.8s
                   all       2093       4887      0.765      0.722       0.73      0.498
 bacterial_leaf_blight         69        125      0.552      0.568      0.515      0.245
bacterial_panicle_blight         45         45      0.755      0.822      0.799      0.591
                 blast        382        947      0.728      0.508       0.58      0.367
     

## C.3 Evaluasi Final pada Data Uji (Test Set) & Kalkulasi F1-Score


In [ ]:
def compute_f1(precision: float, recall: float) -> float:
    if precision + recall > 0:
        return 2 * precision * recall / (precision + recall)
    return 0.0

final_metrics = evaluate_model(best_model, CFG, split="test")

print("=== EVALUASI RESMI DEWAN JURI (TEST SET) ===")
print(f"Test mAP@50    : {final_metrics.box.map50:.4f}")
print(f"Test mAP@50-95 : {final_metrics.box.map:.4f}")

test_precision = final_metrics.box.mp
test_recall = final_metrics.box.mr
test_f1 = compute_f1(test_precision, test_recall)

print(f"Test Precision : {test_precision:.4f}")
print(f"Test Recall    : {test_recall:.4f}")
print(f"Test F1-Score  : {test_f1:.4f}")

# Ekstraksi F1 optimal dari kurva F1 resmi Ultralytics
try:
    f1_curve = final_metrics.box.f1
    mean_f1 = f1_curve.mean(axis=0)
    opt_idx = int(np.argmax(mean_f1))
    opt_f1 = float(mean_f1[opt_idx])
    opt_conf = float(final_metrics.box.curves_results[0][0][opt_idx])
    print(f"Optimal F1 (at conf={opt_conf:.2f}) : {opt_f1:.4f}")
except Exception:
    pass


Ultralytics 8.4.163 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26s summary (fused): 120 layers, 9,469,437 parameters, 0 gradients, 20.8 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 16.2±14.8 MB/s, size: 30.9 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/working/train/labels/test... 1054 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1054/1054 801.6it/s 1.3s
val: /kaggle/working/train/images/test/rtf_43_jpg.rf.8f5cea06bb2ce124569df84dec45f100.jpg: 1 duplicate labels removed
val: New cache created: /kaggle/working/train/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 66/66 3.9it/s 16.8s
                   all       1054       2669      0.745      0.708      0.712      0.509
 bacterial_leaf_blight         29         59      0.441      0.4